In [5]:
import torchvision.transforms as transforms
import torch.utils.data as data
import torchvision.datasets as datasets
import torch

# Constante
BATCH_SIZE = 32
IMG_SIZE = (256, 256) # Je redimensionne car en moyenne elle font 360x460, 256x256 est le choix le plus optimale
ROOT = '../data/clean/'

transform = transforms.Compose([transforms.Resize(IMG_SIZE), transforms.ToTensor(),transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])])
train_data = datasets.ImageFolder(root=f'{ROOT}train', transform=transform)
val_data = datasets.ImageFolder(root=f'{ROOT}val', transform=transform)
test_data = datasets.ImageFolder(root=f'{ROOT}test', transform=transform)

# Loaders
train_loader = data.DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = data.DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = data.DataLoader(test_data, batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [6]:
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        layers = []
        in_channels = 3 # 3 canaux pour les images RGB
        out_channels = 16 #je commence avec 16 filtres pour la convolutions
        for _ in range(5): #5 couches pour arrivés a une taille d'images de 8x8
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            in_channels = out_channels
            out_channels *= 2
        self.features= nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear((out_channels//2)*8*8, 256), # 256 filtres et une taille d'image de 8x8
            nn.ReLU(),
            nn.Linear(256, 2) # 2 classes (dog et cat)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x